# Pipe handling maintenance: visible LLM writing rounds

Choose **GPU T4 × 2** and **Internet On** in Kaggle settings. Upload only this notebook;
the Python files are embedded and model files download automatically.

**Work through one model section at a time.** Each section has five small cells:
**load + plan → draft → reflection → your feedback + revision → close**.
Every actual generation shows its complete output, time, token usage and sampled peak VRAM.
Read the draft, edit the feedback cell, and run it. Repeat **reflection → feedback** as often
as useful. The model stays loaded between these cells. Run the close cell when finished;
starting another model also closes the previous one automatically.

There is **one resident LLM on one GPU**, with the other T4 idle. There is no parallel benchmark,
automatic rewriting loop, or hidden generation. “Run all” demonstrates one shorter revision
per enabled model using the example feedback; manual cell execution lets you steer each round.

All generation texts, exact inputs, per-pass metrics and continuous VRAM samples are also saved
under `/kaggle/working/pipe_maintenance_llm_lab/artifacts`. Downloads and runtime files use scratch
storage outside `/kaggle/working`, so they do not inflate the saved result archive.


## What your completed run established

These are **measured total-device peaks from your supplied completed notebook**, with a 32,768-token
active context and the KV cache on the CPU. They are not estimates of checkpoint file sizes.

| Candidate | Quantization | Observed peak GB | Result in your earlier run |
|---|---|---:|---|
| Qwen3.5 9B | Q5_K_M | 6.651 | Completed; first candidate to inspect here |
| Gemma 4 12B Instruct | Q4_K_M | 7.822 | Completed; compare prose and coverage |
| Ministral 3 8B Instruct | Q6_K | 7.295 | One review exhausted its 2,048-token limit |
| Qwen3.5 4B | Q6_K | Unmeasured | Download disconnected; no inference result |

Start with **Qwen 9B and Gemma 12B**. Qwen was quicker on your directional cases; both followed
shorter/longer/keep in those cases. Numeric ranges still sometimes failed, so the notebook measures
adherence instead of assuming it. Model self-reviews are useful feedback, not independent accuracy scores.

An **optional Qwen3 14B Q4_K_M** section is disabled by default. Its pinned GGUF is **9.002 GB**,
before GPU overhead. Automatic fitting may place some layers on the CPU; full GPU residency and
superior writing quality are not assumed. Its native context is 32,768 tokens. This is a larger
parameter comparison, not a replacement recommendation before measurement.

Model references: [Qwen3.5 9B](https://huggingface.co/Qwen/Qwen3.5-9B),
[Gemma 4 12B](https://huggingface.co/google/gemma-4-12B-it),
[Ministral 3 8B](https://huggingface.co/mistralai/Ministral-3-8B-Instruct-2512),
[Qwen3.5 4B](https://huggingface.co/Qwen/Qwen3.5-4B),
[Qwen3 14B model card](https://huggingface.co/Qwen/Qwen3-14B),
[pinned 14B quantization](https://huggingface.co/unsloth/Qwen3-14B-GGUF/blob/d2f0b1e52b07ef479c3a7bd2f0906ed7678cc7f4/Qwen3-14B-Q4_K_M.gguf).


## 1. Configuration

**10.5 GB = 10,500,000,000 bytes of total GPU usage**, including other applications.
The guard stops inference at **9.0 GB**, retaining 1.5 GB of headroom. Continuous NVML samples
are taken every 10 ms, including loading, idle time, prompt processing and decoding, with fresh
boundaries for every generation. These are sampled peaks: a reactive guard cannot prove that
no shorter allocation transient exceeded the ceiling.

The **32K active context** includes the input plus output allowance. Its CPU KV cache keeps context
memory off the GPU and costs host RAM and speed. Each generation counts the actual native prompt
tokens and rejects an oversized request without truncating its samples or sources.
Enable `RUN_CONTEXT_PROBE` for a visible retrieval test near 24K actual input tokens per model.


In [1]:
import base64
import csv
import gzip
import hashlib
import html
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

from IPython.display import HTML, FileLink, display

# Re-running configuration first releases a previous live session in this kernel.
if "lab" in globals():
    globals()["lab"].close()

ROOT = Path("/kaggle/working/pipe_maintenance_llm_lab")
SCRATCH = Path("/kaggle/temp/pipe_maintenance_llm_lab")
SOURCE_ROOT = ROOT / "source"
ARTIFACTS = ROOT / "artifacts"
DEPENDENCIES = SCRATCH / "python_dependencies"
for directory in (SOURCE_ROOT, ARTIFACTS, DEPENDENCIES):
    directory.mkdir(parents=True, exist_ok=True)

GPU_INDEX = 0                    # Physical GPU index: choose 0 or 1.
CONTEXT_TOKENS = 32768
GENERATION_TOKENS = 4096         # Output token allowance, not a word target.
REVIEW_TOKENS = 4096             # Increased from the exhausted 2048-token review.
THREADS = min(4, len(os.sched_getaffinity(0)))
RUN_CONTEXT_PROBE = False       # Optional; tests ~24K native input tokens.
RUN_LARGER_MODEL = False         # Optional 14B comparison; keep False initially.
MODEL_PATHS = {}                 # Optional pinned GGUF paths in /kaggle/input.


In [2]:
# Embedded, SHA256-verified source. No separate Python upload is needed.
EMBEDDED_SOURCE_DIGEST = '0914c04d9f76e1bb7e4a472a5a6096d3a43e8dc69b7eb9ca00b94a2e29b26238'
EMBEDDED_MANIFEST = {'llm_writing/__init__.py': '8142da5e4a9347e611960e3dd85f86f756eb5796a464dbb7c012fca4ca3212ea', 'llm_writing/assets.py': '30dff7d907fcfaad3f031ea436fbdf17f3fe237a7640edd2008cb2fdbcc7991c', 'llm_writing/cli.py': '5189058bde9f00e742d03ec9ae85726969268ecf00e4fb6d1bcde24a1367dc00', 'llm_writing/pipeline.py': '636afb0782062b5ee373d5dcfde9d3bb90fb6320d55ba1a279afaf67b3d262ce', 'llm_writing/runtime.py': '3e52ed2d47d267eb37be7023b6cd9371baba69ffaa811d6322091bbd47418208', 'llm_writing/interactive.py': '705274f9f110f37b0aaf6407ab9e28f7e9f91647bea962e6c935a7eeb812b955', 'llm_writing/kaggle_setup.py': '05c03bdf1042aaa1705be8436c787581aba133bbea45c59994ce343c051c4aaa', 'llm_writing/kaggle_worker.py': '6d0dc0b85fef2bc9e79f071bbe87701d15b8e69b756ac8060da7898dbffa023a', 'llm_writing/model_manifest.json': 'e8337a05428ea00677f699d5a1b1ea1ab50467737c80d86ae5cb8806708ca693', 'llm_writing/optional_model_manifest.json': 'c4cec35ec708ee508194dc6c4fd77d162d3956b5b11bd33f4d55fa654fcb8781', 'llm_writing/qwen35_9b.py': 'd3aad165911cd5a184ba6077703348a251b02187b6b48a8e773401b7312754d0', 'llm_writing/gemma4_12b.py': '30b59459062104aaff0ab5eabdd67904fd2a36af1d71fc5c97068f44a9a6bfaf', 'llm_writing/ministral3_8b.py': '0c5a3bab5f12ce0953c3e8984c8c2a489469f13bfb7b3969a061e2aae24a0da1', 'llm_writing/qwen35_4b.py': '9c6582cbdd64e60ec43f3537bd490906eb6763f82fa3e761873ed90cd7239a41', 'llm_writing/qwen3_14b.py': 'ae242758be41461f2a177f3986189a0fce5131752b11fdccef885a620e5740e6'}
EMBEDDED_PAYLOAD = (
"..."
)
# A reused Kaggle kernel may still have the older notebook's package cached.
previous_interactive = sys.modules.get("llm_writing.interactive")
if previous_interactive is not None:
    owner = previous_interactive.InteractiveLab
    pending = getattr(owner, "pending_cleanup", None)
    if pending is not None:
        pending()
        owner.pending_cleanup = None
    previous_session = getattr(owner, "active", None)
    if previous_session is not None:
        previous_session.close()
        owner.active = None
for module_name in list(sys.modules):
    if module_name == "llm_writing" or module_name.startswith("llm_writing."):
        sys.modules.pop(module_name, None)
__import__("importlib").invalidate_caches()
embedded = json.loads(gzip.decompress(base64.b64decode(EMBEDDED_PAYLOAD, validate=True)))
if set(embedded) != set(EMBEDDED_MANIFEST):
    raise RuntimeError("Embedded source manifest mismatch")
if hashlib.sha256(json.dumps(EMBEDDED_MANIFEST, sort_keys=True, separators=(",", ":")).encode()).hexdigest() != EMBEDDED_SOURCE_DIGEST:
    raise RuntimeError("Embedded source digest mismatch")
for relative, encoded in embedded.items():
    relative_path = Path(relative)
    if relative_path.is_absolute() or ".." in relative_path.parts or "\\" in relative:
        raise RuntimeError("Unsafe embedded path")
    data = base64.b64decode(encoded, validate=True)
    if hashlib.sha256(data).hexdigest() != EMBEDDED_MANIFEST[relative]:
        raise RuntimeError("Embedded file checksum mismatch: " + relative)
    destination = SOURCE_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(data)
sys.path.insert(0, str(SOURCE_ROOT))
(ARTIFACTS / "source_manifest.json").write_text(json.dumps(EMBEDDED_MANIFEST, indent=2))
print("Extracted", len(embedded), "Python/configuration files; source", EMBEDDED_SOURCE_DIGEST[:12])


Extracted 15 Python/configuration files; source 0914c04d9f76


## 2. Lightweight setup

This notebook uses the existing Kaggle kernel and a small local dependency directory.
**There is no venv, `ensurepip`, Transformers, or PyTorch installation.** It installs only
NVML bindings and `wrapt`; child Python startup is checked with that directory available.
The pinned native CUDA server is the same build that worked in your run. Existing runtime
downloads in the earlier scratch directory are reused after verification.

A disconnected model download retains its partial file and retries from the saved byte offset.
If networking still fails after five attempts, rerun that model's load cell to resume.


In [3]:
requirements = ["nvidia-ml-py==12.575.51", "wrapt==1.17.3"]
install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-deps",
     "--only-binary=:all:", "--upgrade", "--target", str(DEPENDENCIES), *requirements],
    capture_output=True, text=True, timeout=300)
install_log = ARTIFACTS / "dependency_install.log"
install_log.write_text(install.stdout + "\n" + install.stderr, encoding="utf-8")
if install.returncode:
    raise RuntimeError("Dependency installation failed: " + str(install_log) + "\n" + install.stderr[-3000:])
sys.path.insert(0, str(DEPENDENCIES))
os.environ["PYTHONPATH"] = os.pathsep.join([str(DEPENDENCIES), str(SOURCE_ROOT)])
startup = subprocess.run([sys.executable, "-c",
    "import wrapt,pynvml; print('STARTUP_OK: wrapt and NVML imported')"],
    capture_output=True, text=True, timeout=30)
(ARTIFACTS / "startup_probe.log").write_text(startup.stdout + "\n" + startup.stderr)
if startup.returncode or "Error in sitecustomize" in startup.stdout + startup.stderr:
    raise RuntimeError("Python startup failed: " + startup.stdout + startup.stderr)
print(startup.stdout.strip())

from llm_writing.kaggle_setup import ensure_runtime  # noqa: E402
SERVER, SETUP_REPORT = ensure_runtime(Path("/kaggle/temp/worksite_llm_benchmark_runtime/llama_cpp"))
(ARTIFACTS / "runtime_setup.json").write_text(json.dumps(SETUP_REPORT, indent=2))
for gpu in SETUP_REPORT["environment"]["gpus"]:
    print("GPU", gpu["index"], gpu["name"], "capacity", round(gpu["total_bytes"] / 1e9, 3), "GB")
print("Native server ready:", SERVER)


STARTUP_OK: wrapt and NVML imported
llama.cpp b11384: prebuilt CUDA provisioning
GPU 0 Tesla T4 capacity 16.106 GB
GPU 1 Tesla T4 capacity 16.106 GB
Native server ready: /kaggle/temp/worksite_llm_benchmark_runtime/llama_cpp/release/binary/llama-b11384/llama-server


## 3. The brief, optional samples and reference material

Edit the title, instructions, points, samples and approximate initial range here. Samples
demonstrate voice and structure; they do not become verified procedures. Set `samples=()` to
omit them. Add real approved references as `Source` objects, or use `Source("manual", Path(...).read_text())`.
These examples and the reference below are **synthetic training material**, not an approved
maintenance procedure. They deliberately omit equipment limits, torque values and isolation sequences.


In [4]:
from llm_writing.pipeline import Source, WritingRequest

SAMPLE_1 = """Reliable pipe handling starts with equipment that is maintained consistently.
Before maintenance is planned, the team reviews the equipment's condition, the reported
defects and the manufacturer's maintenance instructions. Everyone should understand which
parts of the system will be worked on and who is responsible for the task.

Maintenance must follow the approved site procedure, including isolation and control of
stored energy. Once the equipment is in the approved condition for the work, authorized
personnel inspect the relevant rollers, gripping surfaces, guards, hoses and connections.
Leaks, unusual wear and damaged components are recorded and referred for assessment rather
than treated as normal. Lubrication and replacement work follow the specified maintenance
requirements. Before the system returns to service, the responsible person confirms that
the required checks and records are complete."""

SAMPLE_2 = """A good maintenance record explains what was found, what was done and what still
needs attention. For a pipe handling system, useful observations may concern the condition
of guides and rollers, gripping components, hydraulic connections, protective guards or
control devices. Descriptions should be clear enough for the next person to understand the
defect without guessing.

Work is carried out by authorized personnel using the applicable manufacturer instructions
and site maintenance procedure. Isolation and stored energy controls are part of that
procedure. A component that needs further assessment stays out of service as directed by
the responsible person. The handover includes outstanding defects, the work completed and
the results of the prescribed checks. Return to service follows the approved process;
finishing a repair does not by itself establish that the entire system is ready."""

REFERENCE = """FICTIONAL DEMO REFERENCE: Maintenance of pipe handling systems.
Use the relevant manufacturer's maintenance instructions, site procedure and authorized
maintenance personnel. Plan the work from condition reports and outstanding defects.
Follow the approved procedure for isolation and control of stored energy; no specific
isolation sequence is supplied in this reference. Inspect the applicable rollers, guides,
gripping components, guards, hydraulic hoses and connections, and control/safety devices
as specified for the actual system. Report leaks, wear, loose or damaged components and
abnormal operation for assessment. Keep equipment with unresolved defects out of service
as directed by the responsible person. Lubrication, adjustment and replacement follow
manufacturer requirements; this reference supplies no torque, pressure or service interval.
Prescribed functional checks and authorized handover/return to service follow the approved
procedure. Record condition findings, maintenance completed, checks and outstanding defects.
Never infer equipment-specific steps from this general training reference."""

BRIEF = WritingRequest(
    title="Maintenance of pipe handling systems",
    instructions="Write practical spoken training for workers and supervisors. Explain why "
                 "maintenance matters, how the work is planned, what condition checks cover, "
                 "and how defects and return to service are handled. Use connected prose.",
    required_points=(
        "Plan maintenance using condition reports and manufacturer instructions.",
        "Use authorized personnel and the approved isolation and stored energy procedure.",
        "Cover rollers, guides and gripping components.",
        "Cover guards, hydraulic hoses/connections and applicable control or safety devices.",
        "Report defects and keep unresolved equipment out of service as directed.",
        "Follow manufacturer requirements for lubrication, adjustment and replacement.",
        "Complete prescribed checks, records and authorized return to service.",
    ),
    samples=(SAMPLE_1, SAMPLE_2),
    sources=(Source("maintenance-demo", REFERENCE, "Synthetic demo reference; replace for real equipment"),),
    min_words=350, max_words=500,
    revision_passes=0, length_passes=0,  # Stages are manually controlled in this notebook.
)
BRIEF.validate()
print("Title:", BRIEF.title, "| initial approximate range:", BRIEF.word_range())


Title: Maintenance of pipe handling systems | initial approximate range: (350, 500)


## 4. Display and single-model controller

Green/failed labels describe output acceptance, not factual approval. Every pass is displayed
even if it is truncated and rejected. An unsuccessful revision retains the previous good text.
No numeric word target is invented for direction-only feedback. Later length feedback clears
the previous numeric range unless you supply a new `word_range` explicitly.


In [5]:
from llm_writing.interactive import InteractiveLab

def show_generation(event):
    peak = event.get("total_device_peak_gb")
    peak_label = f"{peak:.3f} GB" if peak is not None else "unavailable"
    usage = event.get("usage", {})
    label = (f"{event['profile']} · pass {event['round_id']} · {event['stage']} · {event['status']}"
             f" · {event['latency_seconds']:.1f} s · peak total GPU VRAM {peak_label}"
             f" · input/output tokens {usage.get('prompt_tokens')}/{usage.get('completion_tokens')}")
    text = event.get("text") or "[No output returned]"
    display(HTML("<h4>" + html.escape(label) + "</h4><pre style='white-space:pre-wrap;"
                 "overflow-wrap:anywhere;font-size:14px'>" + html.escape(text) + "</pre>"))
    if event.get("error"):
        print("Rejected pass:", event["error"]["message"])

def show_table(rows):
    if not rows:
        print("No completed drafts to compare yet.")
        return
    keys = list(rows[0])
    header = "".join("<th>" + html.escape(k) + "</th>" for k in keys)
    body = "".join("<tr>" + "".join("<td>" + html.escape(str(row.get(k))) + "</td>" for k in keys)
                   + "</tr>" for row in rows)
    display(HTML("<div style='overflow-x:auto'><table border='1' cellpadding='6'><tr>"
                 + header + "</tr>" + body + "</table></div>"))

lab = InteractiveLab(SERVER, SCRATCH / "models", ARTIFACTS, render=show_generation,
                     context_size=CONTEXT_TOKENS, generation_tokens=GENERATION_TOKENS,
                     review_tokens=REVIEW_TOKENS, threads=THREADS)
print("Controller ready. Each following cell makes its explicitly named generation only.")


Controller ready. Each following cell makes its explicitly named generation only.


## How to steer a round

In each model's feedback cell, change:

```python
FEEDBACK = dict(length_direction="shorter", instructions="Make it noticeably tighter.", word_range=None)
# Or: "longer", "keep"; instructions can specify how much change and other editorial changes.
# Optional approximate bounds: word_range=(250, 350).
# Guidance only: length_direction=None, instructions="Give more attention to defect records."
```

The new word count is compared with the **immediately preceding draft**. “Keep” is measured
within ±10%; “shorter”/“longer” measure the sign of the change. Read the text to judge whether
the amount of change is useful. Rerun the reflection cell to review the new text, then edit
and rerun feedback. Every pass retains the original title, points, samples and references;
editorial feedback accumulates, while the latest length policy takes precedence.

If a pass exhausts its output tokens, its partial output is shown and saved but not accepted.
Increase `session.pipeline.review_tokens` for a reflection or `generation_tokens` for prose,
then rerun that cell. The context admission check still applies. A telemetry or VRAM failure
requires closing that session and inspecting its saved native log before retrying.


## Model 1: Qwen3.5 9B (Q5_K_M)

Run these cells in order. Stop before the close cell while experimenting with feedback.
A: load + plan. B: draft. C: reflection. D: edit feedback + revise. E: optional context test + close.
Repeat C and D while editing. Run C before E if you want a review of the final revision.


In [6]:
# A. Load this model and generate its plan. Previous models are closed automatically.
model_1 = None
if True:
    model_1 = lab.start('qwen35_9b', gpu_index=GPU_INDEX,
                           model_path=MODEL_PATHS.get('qwen35_9b'))
    model_1.plan(BRIEF)
else:
    print("Optional larger model disabled; no weights downloaded.")


Model download: 9.5%
Model download: 23.1%
Model download: 33.7%
Model download: 47.6%
Model download: 59.7%
Model download: 71.4%
Model download: 80.8%
Model download: 95.5%
Verifying SHA256 of Qwen3.5-9B-Q5_K_M.gguf
Loading Qwen3.5 9B (Q5_K_M) on GPU 0; context 32768; guard 9.0 GB
Ready. CPU KV cache; one model and one GPU only. Run plan, draft, reflection and feedback cells.
ROUND qwen35_9b/interactive/plan: peak=6.636634112 GB, status=ok
Confirmed GPU tensor layers: 33/33


In [7]:
# B. Generate the first full text.
if model_1 is not None:
    _ = model_1.draft()


ROUND qwen35_9b/interactive/draft: peak=6.651314176 GB, status=ok



Measured words: 387; previous baseline: None; change: None; range: [350, 500]; range met: True; direction met: None
Edit the feedback and run revise again; reflect() reviews the current version.


In [8]:
# C. Reflect on the CURRENT text. Rerun after each revision if useful.
if model_1 is not None:
    _ = model_1.reflect()


ROUND qwen35_9b/interactive/reflection: peak=6.651314176 GB, status=ok


In [9]:
# D. Edit this OUTSIDE feedback for each round, then generate one full revision.
FEEDBACK_1 = dict(
    length_direction="shorter",  # "longer", "keep", or None for guidance only.
    instructions="Remove repetition and tighten the phrasing while preserving every required point.",
    word_range=None,            # Optional example: (250, 350). None clears the old range.
)
if model_1 is not None:
    _ = model_1.revise(**FEEDBACK_1)


ROUND qwen35_9b/interactive/revision: peak=6.651314176 GB, status=ok



Measured words: 299; previous baseline: 387; change: -88; range: None; range met: None; direction met: True
Edit the feedback and run revise again; reflect() reviews the current version.


In [10]:
# E. Finish this model and release its GPU memory when satisfied.
if model_1 is not None:
    if RUN_CONTEXT_PROBE:
        _ = model_1.context_probe(24000)
    model_1.close()
    show_table([row for row in lab.comparison() if row["model"] == 'qwen35_9b'])


qwen35_9b: closed; texts and metrics saved in /kaggle/working/pipe_maintenance_llm_lab/artifacts/qwen35_9b/d2cd5c294829


model,run,words,direction_met,range_met,passes,failed_passes,generation_peak_GB,lifetime_peak_GB,generation_seconds,missing_points_model_review,current_text_reviewed,closed,GPU_tensor_layers,total_tensor_layers,output
qwen35_9b,d2cd5c294829,299,True,None,4,0,6.651314176,6.651314176,207.5,None,False,True,33,33,/kaggle/working/pipe_maintenance_llm_lab/artifacts/qwen35_9b/d2cd5c294829


## Model 2: Gemma 4 12B Instruct (Q4_K_M)

Run these cells in order. Stop before the close cell while experimenting with feedback.
A: load + plan. B: draft. C: reflection. D: edit feedback + revise. E: optional context test + close.
Repeat C and D while editing. Run C before E if you want a review of the final revision.


In [11]:
# A. Load this model and generate its plan. Previous models are closed automatically.
model_2 = None
if True:
    model_2 = lab.start('gemma4_12b', gpu_index=GPU_INDEX,
                           model_path=MODEL_PATHS.get('gemma4_12b'))
    model_2.plan(BRIEF)
else:
    print("Optional larger model disabled; no weights downloaded.")


Model download: 2.3%
Model download: 9.3%
Model download: 11.0%
Model download: 16.7%
Model download: 28.2%
Model download: 38.1%
Model download: 47.5%
Model download: 53.5%
Model download: 60.4%
Model download: 69.6%
Model download: 80.5%
Model download: 88.8%
Model download: 94.1%
Model download: 96.8%
Verifying SHA256 of gemma-4-12b-it-Q4_K_M.gguf
Loading Gemma 4 12B Instruct (Q4_K_M) on GPU 0; context 32768; guard 9.0 GB
Ready. CPU KV cache; one model and one GPU only. Run plan, draft, reflection and feedback cells.
ROUND gemma4_12b/interactive/plan: peak=7.80894208 GB, status=ok
Confirmed GPU tensor layers: 49/49


In [12]:
# B. Generate the first full text.
if model_2 is not None:
    _ = model_2.draft()


ROUND gemma4_12b/interactive/draft: peak=7.821524992 GB, status=ok



Measured words: 404; previous baseline: None; change: None; range: [350, 500]; range met: True; direction met: None
Edit the feedback and run revise again; reflect() reviews the current version.


In [13]:
# C. Reflect on the CURRENT text. Rerun after each revision if useful.
if model_2 is not None:
    _ = model_2.reflect()


ROUND gemma4_12b/interactive/reflection: peak=7.821524992 GB, status=ok


In [14]:
# D. Edit this OUTSIDE feedback for each round, then generate one full revision.
FEEDBACK_2 = dict(
    length_direction="shorter",  # "longer", "keep", or None for guidance only.
    instructions="Remove repetition and tighten the phrasing while preserving every required point.",
    word_range=None,            # Optional example: (250, 350). None clears the old range.
)
if model_2 is not None:
    _ = model_2.revise(**FEEDBACK_2)


ROUND gemma4_12b/interactive/revision: peak=7.821524992 GB, status=ok



Measured words: 358; previous baseline: 404; change: -46; range: None; range met: None; direction met: True
Edit the feedback and run revise again; reflect() reviews the current version.


In [15]:
# E. Finish this model and release its GPU memory when satisfied.
if model_2 is not None:
    if RUN_CONTEXT_PROBE:
        _ = model_2.context_probe(24000)
    model_2.close()
    show_table([row for row in lab.comparison() if row["model"] == 'gemma4_12b'])


gemma4_12b: closed; texts and metrics saved in /kaggle/working/pipe_maintenance_llm_lab/artifacts/gemma4_12b/dfa0b5733195


model,run,words,direction_met,range_met,passes,failed_passes,generation_peak_GB,lifetime_peak_GB,generation_seconds,missing_points_model_review,current_text_reviewed,closed,GPU_tensor_layers,total_tensor_layers,output
gemma4_12b,dfa0b5733195,358,True,None,4,0,7.821524992,7.821524992,369.4,None,False,True,49,49,/kaggle/working/pipe_maintenance_llm_lab/artifacts/gemma4_12b/dfa0b5733195


## Model 3: Ministral 3 8B Instruct 2512 (Q6_K)

Run these cells in order. Stop before the close cell while experimenting with feedback.
A: load + plan. B: draft. C: reflection. D: edit feedback + revise. E: optional context test + close.
Repeat C and D while editing. Run C before E if you want a review of the final revision.


In [16]:
# A. Load this model and generate its plan. Previous models are closed automatically.
model_3 = None
if True:
    model_3 = lab.start('ministral3_8b', gpu_index=GPU_INDEX,
                           model_path=MODEL_PATHS.get('ministral3_8b'))
    model_3.plan(BRIEF)
else:
    print("Optional larger model disabled; no weights downloaded.")


Model download: 8.7%
Model download: 15.3%
Model download: 22.7%
Model download: 28.5%
Model download: 35.8%
Model download: 42.8%
Model download: 49.4%
Model download: 56.8%
Model download: 63.2%
Model download: 68.8%
Model download: 76.4%
Model download: 81.1%
Model download: 86.9%
Model download: 93.3%
Verifying SHA256 of Ministral-3-8B-Instruct-2512-Q6_K.gguf
Loading Ministral 3 8B Instruct 2512 (Q6_K) on GPU 0; context 32768; guard 9.0 GB
Ready. CPU KV cache; one model and one GPU only. Run plan, draft, reflection and feedback cells.
ROUND ministral3_8b/interactive/plan: peak=7.29513984 GB, status=ok
Confirmed GPU tensor layers: 35/35


In [17]:
# B. Generate the first full text.
if model_3 is not None:
    _ = model_3.draft()


ROUND ministral3_8b/interactive/draft: peak=7.29513984 GB, status=ok



Measured words: 589; previous baseline: None; change: None; range: [350, 500]; range met: False; direction met: None
Edit the feedback and run revise again; reflect() reviews the current version.


In [18]:
# C. Reflect on the CURRENT text. Rerun after each revision if useful.
if model_3 is not None:
    _ = model_3.reflect()


ROUND ministral3_8b/interactive/reflection: peak=7.29513984 GB, status=ok


In [19]:
# D. Edit this OUTSIDE feedback for each round, then generate one full revision.
FEEDBACK_3 = dict(
    length_direction="shorter",  # "longer", "keep", or None for guidance only.
    instructions="Remove repetition and tighten the phrasing while preserving every required point.",
    word_range=None,            # Optional example: (250, 350). None clears the old range.
)
if model_3 is not None:
    _ = model_3.revise(**FEEDBACK_3)


ROUND ministral3_8b/interactive/revision: peak=7.29513984 GB, status=ok



Measured words: 460; previous baseline: 589; change: -129; range: None; range met: None; direction met: True
Edit the feedback and run revise again; reflect() reviews the current version.


In [20]:
# E. Finish this model and release its GPU memory when satisfied.
if model_3 is not None:
    if RUN_CONTEXT_PROBE:
        _ = model_3.context_probe(24000)
    model_3.close()
    show_table([row for row in lab.comparison() if row["model"] == 'ministral3_8b'])


ministral3_8b: closed; texts and metrics saved in /kaggle/working/pipe_maintenance_llm_lab/artifacts/ministral3_8b/3a576f3e1695


model,run,words,direction_met,range_met,passes,failed_passes,generation_peak_GB,lifetime_peak_GB,generation_seconds,missing_points_model_review,current_text_reviewed,closed,GPU_tensor_layers,total_tensor_layers,output
ministral3_8b,3a576f3e1695,460,True,None,4,0,7.29513984,7.29513984,338.5,None,False,True,35,35,/kaggle/working/pipe_maintenance_llm_lab/artifacts/ministral3_8b/3a576f3e1695


## Model 4: Qwen3.5 4B (Q6_K)

Run these cells in order. Stop before the close cell while experimenting with feedback.
A: load + plan. B: draft. C: reflection. D: edit feedback + revise. E: optional context test + close.
Repeat C and D while editing. Run C before E if you want a review of the final revision.


In [21]:
# A. Load this model and generate its plan. Previous models are closed automatically.
model_4 = None
if True:
    model_4 = lab.start('qwen35_4b', gpu_index=GPU_INDEX,
                           model_path=MODEL_PATHS.get('qwen35_4b'))
    model_4.plan(BRIEF)
else:
    print("Optional larger model disabled; no weights downloaded.")


Model download: 20.6%
Model download: 43.4%
Model download: 60.4%
Model download: 78.3%
Model download: 96.1%
Verifying SHA256 of Qwen3.5-4B-Q6_K.gguf
Loading Qwen3.5 4B (Q6_K) on GPU 0; context 32768; guard 9.0 GB
Ready. CPU KV cache; one model and one GPU only. Run plan, draft, reflection and feedback cells.
ROUND qwen35_4b/interactive/plan: peak=4.29621248 GB, status=ok
Confirmed GPU tensor layers: 33/33


In [22]:
# B. Generate the first full text.
if model_4 is not None:
    _ = model_4.draft()


ROUND qwen35_4b/interactive/draft: peak=4.29621248 GB, status=ok



Measured words: 329; previous baseline: None; change: None; range: [350, 500]; range met: False; direction met: None
Edit the feedback and run revise again; reflect() reviews the current version.


In [23]:
# C. Reflect on the CURRENT text. Rerun after each revision if useful.
if model_4 is not None:
    _ = model_4.reflect()


ROUND qwen35_4b/interactive/reflection: peak=4.298309632 GB, status=ok


In [24]:
# D. Edit this OUTSIDE feedback for each round, then generate one full revision.
FEEDBACK_4 = dict(
    length_direction="shorter",  # "longer", "keep", or None for guidance only.
    instructions="Remove repetition and tighten the phrasing while preserving every required point.",
    word_range=None,            # Optional example: (250, 350). None clears the old range.
)
if model_4 is not None:
    _ = model_4.revise(**FEEDBACK_4)


ROUND qwen35_4b/interactive/revision: peak=4.298309632 GB, status=ok



Measured words: 232; previous baseline: 329; change: -97; range: None; range met: None; direction met: True
Edit the feedback and run revise again; reflect() reviews the current version.


In [25]:
# E. Finish this model and release its GPU memory when satisfied.
if model_4 is not None:
    if RUN_CONTEXT_PROBE:
        _ = model_4.context_probe(24000)
    model_4.close()
    show_table([row for row in lab.comparison() if row["model"] == 'qwen35_4b'])


qwen35_4b: closed; texts and metrics saved in /kaggle/working/pipe_maintenance_llm_lab/artifacts/qwen35_4b/ee133077fedb


model,run,words,direction_met,range_met,passes,failed_passes,generation_peak_GB,lifetime_peak_GB,generation_seconds,missing_points_model_review,current_text_reviewed,closed,GPU_tensor_layers,total_tensor_layers,output
qwen35_4b,ee133077fedb,232,True,None,4,0,4.298309632,4.298309632,122.7,None,False,True,33,33,/kaggle/working/pipe_maintenance_llm_lab/artifacts/qwen35_4b/ee133077fedb


## Model 5: Qwen3 14B (Q4_K_M; experimental CPU/GPU fit)

Optional: set `RUN_LARGER_MODEL=True` in configuration, then run this section. Some layers may remain on the CPU.
A: load + plan. B: draft. C: reflection. D: edit feedback + revise. E: optional context test + close.
Repeat C and D while editing. Run C before E if you want a review of the final revision.


In [26]:
# A. Load this model and generate its plan. Previous models are closed automatically.
model_5 = None
if RUN_LARGER_MODEL:
    model_5 = lab.start('qwen3_14b', gpu_index=GPU_INDEX,
                           model_path=MODEL_PATHS.get('qwen3_14b'))
    model_5.plan(BRIEF)
else:
    print("Optional larger model disabled; no weights downloaded.")


Optional larger model disabled; no weights downloaded.


In [27]:
# B. Generate the first full text.
if model_5 is not None:
    _ = model_5.draft()


In [28]:
# C. Reflect on the CURRENT text. Rerun after each revision if useful.
if model_5 is not None:
    _ = model_5.reflect()


In [29]:
# D. Edit this OUTSIDE feedback for each round, then generate one full revision.
FEEDBACK_5 = dict(
    length_direction="shorter",  # "longer", "keep", or None for guidance only.
    instructions="Remove repetition and tighten the phrasing while preserving every required point.",
    word_range=None,            # Optional example: (250, 350). None clears the old range.
)
if model_5 is not None:
    _ = model_5.revise(**FEEDBACK_5)


In [30]:
# E. Finish this model and release its GPU memory when satisfied.
if model_5 is not None:
    if RUN_CONTEXT_PROBE:
        _ = model_5.context_probe(24000)
    model_5.close()
    show_table([row for row in lab.comparison() if row["model"] == 'qwen3_14b'])


## Compare and save

Compare the same initial brief and the same feedback across models. More manual rounds naturally
increase total time; use each pass's displayed latency when comparing equivalent stages.
Favor complete coverage, sound source use, useful revisions and readable prose, then compare
memory and time. The table reports measured direction/range adherence and failed passes. A final
revision needs its own reflection if you want its current model review recorded.

Save your own quality scores (1–5) after reading the texts. No automatic judge is used to declare
a writing winner. Qwen 9B is the starting recommendation from the earlier run; this maintenance
example lets you test that choice directly against Gemma and the other candidates.


In [31]:
lab.close()
COMPARISON = lab.comparison()
show_table(COMPARISON)
if COMPARISON:
    with (ARTIFACTS / "comparison.csv").open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(COMPARISON[0]))
        writer.writeheader()
        writer.writerows(COMPARISON)

QUALITY_SCORES = {"qwen35_9b": None, "gemma4_12b": None, "ministral3_8b": None,
                  "qwen35_4b": None, "qwen3_14b": None}  # Fill after reading; None means unscored.
(ARTIFACTS / "human_quality_scores.json").write_text(json.dumps(QUALITY_SCORES, indent=2))
ARCHIVE = ROOT / "pipe_maintenance_texts_and_metrics.zip"
with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(ARTIFACTS.rglob("*")):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
print("Saved all generation texts, inputs, feedback histories, metrics and logs:")
display(FileLink(str(ARCHIVE)))


model,run,words,direction_met,range_met,passes,failed_passes,generation_peak_GB,lifetime_peak_GB,generation_seconds,missing_points_model_review,current_text_reviewed,closed,GPU_tensor_layers,total_tensor_layers,output
gemma4_12b,dfa0b5733195,358,True,None,4,0,7.821524992,7.821524992,369.4,None,False,True,49,49,/kaggle/working/pipe_maintenance_llm_lab/artifacts/gemma4_12b/dfa0b5733195
ministral3_8b,3a576f3e1695,460,True,None,4,0,7.29513984,7.29513984,338.5,None,False,True,35,35,/kaggle/working/pipe_maintenance_llm_lab/artifacts/ministral3_8b/3a576f3e1695
qwen35_4b,ee133077fedb,232,True,None,4,0,4.298309632,4.298309632,122.7,None,False,True,33,33,/kaggle/working/pipe_maintenance_llm_lab/artifacts/qwen35_4b/ee133077fedb
qwen35_9b,d2cd5c294829,299,True,None,4,0,6.651314176,6.651314176,207.5,None,False,True,33,33,/kaggle/working/pipe_maintenance_llm_lab/artifacts/qwen35_9b/d2cd5c294829


Saved all generation texts, inputs, feedback histories, metrics and logs:


/kaggle/working/pipe_maintenance_llm_lab/pipe_maintenance_texts_and_metrics.zip